# Блок 3 — Ансамблирование и финальная оценка

| Метод | Описание |
|---|---|
| **Hard Voting** | Каждая модель голосует за класс, побеждает большинство |
| **Soft Voting** | Усреднение вероятностей всех моделей |
| **Weighted Averaging** | Усреднение, взвешенное по F1-macro |
| **Stacking (LogReg)** | Мета-модель: логрегрессия на вероятностях |
| **Stacking (SVM)** | Мета-модель: SVM на вероятностях |
| **Stacking (XGBoost)** | Мета-модель: XGBoost на вероятностях |
| **Stacking (GradientBoosting)** | Мета-модель: Gradient Boosting на вероятностях |
| **Temperature Scaling** | Калибровка уверенности каждой модели |

In [ ]:
import sys, os, json

PROJECT_ROOT = '/kaggle/input/datasets/inexyy/se-analysis' if os.path.exists('/kaggle') else os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

WORKING_DIR = '/kaggle/working' if os.path.exists('/kaggle') else '../results'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120

# Загружаем label_names (с fallback на 7 классов Ekman)
_ln_path = f'{WORKING_DIR}/label_names.json'
if os.path.exists(_ln_path):
    with open(_ln_path) as f:
        label_names = json.load(f)
else:
    label_names = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
print(f'Классы: {label_names}')


In [ ]:
# ── Загрузка предсказаний из HuggingFace Hub ─────────────────────────────
# Структура репо: 02_training_*/models/{model_name}/*.npy
# Stage-1 папки (*_s1) пропускаются — берём только финальные Stage-2.
# Скачивает только .npy + results.json (без весов моделей, ~несколько МБ).
import shutil
from huggingface_hub import snapshot_download

HF_MODELS_REPO = 'Kirillx/ru-text-emotion-classification'
MODELS_BASE     = os.path.join(WORKING_DIR, 'models')

def _preds_ready(base, n=2):
    if not os.path.isdir(base):
        return False
    return sum(
        os.path.exists(os.path.join(base, d, 'test_probs.npy'))
        for d in os.listdir(base)
        if os.path.isdir(os.path.join(base, d))
    ) >= n

if _preds_ready(MODELS_BASE):
    print(f'Предсказания уже на месте: {MODELS_BASE}')
else:
    print(f'Скачиваем из {HF_MODELS_REPO} …')
    hf_dir = snapshot_download(
        repo_id=HF_MODELS_REPO,
        local_dir='/tmp/hf_models',
        allow_patterns=['**/*.npy', '**/results.json', '**/label_names.json'],
    )

    # Динамически находим все папки с test_probs.npy, пропуская Stage-1 (_s1)
    os.makedirs(MODELS_BASE, exist_ok=True)
    found = {}
    for root, dirs, files in os.walk(hf_dir):
        if 'test_probs.npy' in files:
            key = os.path.basename(root)
            if not key.endswith('_s1'):
                found[key] = root

    print(f'Найдено {len(found)} модел(ей): {sorted(found)}')
    for key, src in sorted(found.items()):
        dst = os.path.join(MODELS_BASE, key)
        os.makedirs(dst, exist_ok=True)
        for fname in os.listdir(src):
            if fname.endswith('.npy') or fname == 'results.json':
                shutil.copy2(os.path.join(src, fname), os.path.join(dst, fname))
        print(f'  {key} ✓')

    # label_names.json → WORKING_DIR (берём первый найденный)
    ln_dst = os.path.join(WORKING_DIR, 'label_names.json')
    if not os.path.exists(ln_dst):
        for root, dirs, files in os.walk(hf_dir):
            if 'label_names.json' in files:
                shutil.copy2(os.path.join(root, 'label_names.json'), ln_dst)
                with open(ln_dst) as f:
                    label_names = json.load(f)
                print(f'label_names: {label_names}')
                break

    print('Готово.')


## 1. Загрузка предсказаний

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

MODEL_DIRS = {
    'ruBERT':            f'{WORKING_DIR}/models/rubert',
    'XLM-RoBERTa':       f'{WORKING_DIR}/models/xlmroberta',
    'ruBERT-tiny':       f'{WORKING_DIR}/models/rubert_tiny',
    'ruBERT-large':      f'{WORKING_DIR}/models/rubert_large',
    'ruRoBERTa-large':   f'{WORKING_DIR}/models/ruroberta_large',
    'Aniemore-emotion':  f'{WORKING_DIR}/models/aniemore_emotion',
    'seara-goem':        f'{WORKING_DIR}/models/seara_goem',
}

all_probs  = {}
all_preds  = {}
all_val_f1 = {}
all_labels = None

for name, model_dir in MODEL_DIRS.items():
    probs_path  = os.path.join(model_dir, 'test_probs.npy')
    preds_path  = os.path.join(model_dir, 'test_preds.npy')
    labels_path = os.path.join(model_dir, 'test_labels.npy')

    if not os.path.exists(probs_path):
        print(f'WARNING: {name} — предсказания не найдены в {model_dir}')
        continue

    all_probs[name] = np.load(probs_path)
    all_preds[name] = np.load(preds_path)
    labels = np.load(labels_path)
    if all_labels is None:
        all_labels = labels

    results_path = os.path.join(model_dir, 'results.json')
    if os.path.exists(results_path):
        with open(results_path) as f:
            res = json.load(f)
        all_val_f1[name] = res.get('test_report', {}).get('macro avg', {}).get('f1-score', 0.0)
    else:
        all_val_f1[name] = f1_score(all_labels, all_preds[name], average='macro')

    print(f'{name:<22s}: {len(all_preds[name]):,} примеров | F1-macro={all_val_f1[name]:.4f}')

if len(all_probs) < 2:
    raise RuntimeError('Нужно как минимум 2 обученных модели.')

In [ ]:
from src.evaluation import evaluate_predictions, compare_models

individual_results = []
for name in all_probs:
    m = evaluate_predictions(
        all_labels, all_preds[name], all_probs[name],
        model_name=name, label_names=label_names,
    )
    individual_results.append(m)

df_ind = pd.DataFrame(individual_results).set_index('model')
print(df_ind[['accuracy', 'f1_macro', 'f1_weighted']].round(4).to_string())


## 2. Ансамбли

In [ ]:
from src.ensemble import (
    hard_voting, soft_voting, weighted_averaging,
    stacking_ensemble, evaluate_ensemble,
)

probs_list = list(all_probs.values())
preds_list = list(all_preds.values())
f1_weights = list(all_val_f1.values())

# ── Hard Voting
hv = hard_voting(preds_list)
print(f'Hard Voting   — F1-macro: {f1_score(all_labels, hv, average="macro"):.4f}')

# ── Soft Voting
sv = soft_voting(probs_list)
print(f'Soft Voting   — F1-macro: {f1_score(all_labels, sv, average="macro"):.4f}')

# ── Weighted Averaging
wa = weighted_averaging(probs_list, f1_weights)
print(f'Weighted Avg  — F1-macro: {f1_score(all_labels, wa, average="macro"):.4f}')


In [ ]:
# ── Stacking
# Для мета-обучения берём val-предсказания (без утечки данных)
val_probs_list = []
val_labels_arr = None

for name, model_dir in MODEL_DIRS.items():
    if name not in all_probs:
        continue
    vpath = os.path.join(model_dir, 'val_probs.npy')
    if os.path.exists(vpath):
        val_probs_list.append(np.load(vpath))
        if val_labels_arr is None:
            val_labels_arr = np.load(os.path.join(model_dir, 'val_labels.npy'))
    else:
        n = len(all_probs[name]) // 2
        val_probs_list.append(all_probs[name][:n])
        if val_labels_arr is None:
            val_labels_arr = all_labels[:n]

n_half = len(all_labels) // 2
test_probs_half  = [p[n_half:] for p in list(all_probs.values())]
test_labels_half = all_labels[n_half:]

print('Stacking — Logistic Regression:')
st_lr, _ = stacking_ensemble(val_probs_list, val_labels_arr, test_probs_half, meta_learner='logistic')
print(f'  F1-macro: {f1_score(test_labels_half, st_lr, average="macro"):.4f}')

print('\nStacking — SVM:')
st_svm, _ = stacking_ensemble(val_probs_list, val_labels_arr, test_probs_half, meta_learner='svm')
print(f'  F1-macro: {f1_score(test_labels_half, st_svm, average="macro"):.4f}')

print('\nStacking — XGBoost:')
st_xgb, _ = stacking_ensemble(val_probs_list, val_labels_arr, test_probs_half, meta_learner='xgboost')
print(f'  F1-macro: {f1_score(test_labels_half, st_xgb, average="macro"):.4f}')

print('\nStacking — Gradient Boosting:')
st_gb, _ = stacking_ensemble(val_probs_list, val_labels_arr, test_probs_half, meta_learner='gradient_boosting')
print(f'  F1-macro: {f1_score(test_labels_half, st_gb, average="macro"):.4f}')

## 2b. Калибровка температуры (Temperature Scaling)

In [ ]:
from src.ensemble import fit_temperature, temperature_scaling
from sklearn.metrics import log_loss

print('Подбор температуры на val-предсказаниях:')
calibrated_probs = {}
temperatures = {}

for name, model_dir in MODEL_DIRS.items():
    if name not in all_probs:
        continue
    val_probs_path  = os.path.join(model_dir, 'val_probs.npy')
    val_labels_path = os.path.join(model_dir, 'val_labels.npy')
    if not os.path.exists(val_probs_path):
        print(f'  {name}: val_probs.npy не найден, пропускаем')
        continue

    vp = np.load(val_probs_path)
    vl = np.load(val_labels_path).astype(int)

    # Восстанавливаем логиты из softmax-вероятностей (обратное через log)
    # Примечание: fit_temperature принимает логиты — берём log(prob) как приближение
    import scipy.special
    logits_approx = np.log(np.clip(vp, 1e-9, 1.0))

    T = fit_temperature(logits_approx, vl)
    temperatures[name] = T
    cal_probs = temperature_scaling(logits_approx, T)
    calibrated_probs[name] = cal_probs

    nll_before = log_loss(vl, vp)
    nll_after  = log_loss(vl, cal_probs)
    print(f'  {name:<20s}: T={T:.3f} | NLL before={nll_before:.4f} → after={nll_after:.4f}')

print('\nКалиброванный soft voting:')
if calibrated_probs:
    from src.ensemble import soft_voting_proba
    cal_list = [calibrated_probs[n] for n in all_probs if n in calibrated_probs]
    test_cal_list = [all_probs[n] for n in all_probs if n in calibrated_probs]
    sv_cal = np.argmax(soft_voting_proba(test_cal_list), axis=-1)
    from sklearn.metrics import f1_score as _f1
    print(f'  F1-macro: {_f1(all_labels, sv_cal, average="macro"):.4f}')


## 3. Сравнение всех методов

In [ ]:
ensemble_dict = {
    'Hard Voting':        hv,
    'Soft Voting':        sv,
    'Weighted Averaging': wa,
}
df_ens = evaluate_ensemble(all_labels, ensemble_dict)

df_stk = evaluate_ensemble(
    test_labels_half,
    {
        'Stacking LogReg':    st_lr,
        'Stacking SVM':       st_svm,
        'Stacking XGBoost':   st_xgb,
        'Stacking GradBoost': st_gb,
    },
)

print('=== Ансамбли (полная тест. выборка) ===')
print(df_ens.round(4).to_string())
print('\n=== Стекинг (половина тест. выборки) ===')
print(df_stk.round(4).to_string())


In [ ]:
best_name = df_ens['f1_macro'].idxmax()
best_preds = ensemble_dict[best_name]
best_metrics = evaluate_predictions(
    all_labels, best_preds, model_name=f'Ensemble ({best_name})', label_names=label_names
)
all_rows = individual_results + [best_metrics]
compare_models(all_rows, save_path=f'{WORKING_DIR}/model_comparison.png')
print(f'\nЛучший ансамбль: {best_name} | F1-macro: {best_metrics["f1_macro"]:.4f}')


In [ ]:
from src.evaluation import plot_confusion_matrix
plot_confusion_matrix(
    all_labels, best_preds,
    model_name=f'Best: {best_name}',
    label_names=label_names,
    save_path=f'{WORKING_DIR}/cm_best_ensemble.png',
)


## 4. Итоговый отчёт

In [ ]:
from sklearn.metrics import classification_report

print('\n' + '='*65)
print('ИТОГОВЫЙ ОТЧЁТ — Классификация эмоций (Ekman, 7 классов)')
print('='*65)
print('\n── Индивидуальные модели ──')
print(df_ind[['accuracy', 'f1_macro', 'f1_weighted']].round(4).to_string())
print('\n── Ансамбли (полная тест. выборка) ──')
print(df_ens.round(4).to_string())
print('\n── Стекинг (половина тест. выборки) ──')
print(df_stk.round(4).to_string())
print(f'\n── Лучший ансамбль: {best_name} ──')
print(classification_report(all_labels, best_preds, target_names=label_names))

summary = {
    'task': 'emotion classification (Ekman 7-class)',
    'label_names': label_names,
    'individual_models': df_ind.round(4).to_dict(),
    'ensemble_methods': df_ens.round(4).to_dict(),
    'stacking_methods': df_stk.round(4).to_dict(),
    'best_ensemble': best_name,
    'best_f1_macro': round(float(best_metrics['f1_macro']), 4),
}
with open(f'{WORKING_DIR}/final_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f'\nSummary сохранён: {WORKING_DIR}/final_summary.json')
